CAN BE IGNORED

This notebook contains different testings of api retrieval and SPARQL Queries (not important for the actual backend/app)

In [1]:
import httpx

WIKIDATA_API = "https://www.wikidata.org/w/api.php"

HEADERS = {
    "User-Agent": "ebruhn/1.0 (mailto:s246070@dtu.dk)",
    "Accept": "application/json",
}

def search_wikidata(name, limit=5):
    params = {
        "action": "wbsearchentities",
        "search": name,
        "language": "en",
        "type": "item",
        "format": "json",
        "limit": limit,
    }

    response = httpx.get(
        WIKIDATA_API,
        params=params,
        headers=HEADERS,
        timeout=30,
    )
    response.raise_for_status()

    return response.json()["search"]

In [2]:
results = search_wikidata("Albert Einstein")

for result in results:
    print(
        result["id"],
        result.get("label"),
        result.get("description")
    )

Q937 Albert Einstein German-born theoretical physicist (1879–1955)
Q2030894 Albert Einstein College of Medicine private medical school in New York City, NY
Q2253683 Albert Einstein train service
Q21200226 Albert Einstein Wikimedia permanent duplicate item
Q47513150 Albert Einstein painting by Max Westfield


In [3]:
results = search_wikidata("Albert Einstein")

if results:
    qid = results[0]["id"]
    print(qid)

Q937


In [4]:
def find_qid(name):
    results = search_wikidata(name)

    if not results:
        return None

    for result in results:
        print(
            f"{result['id']}: "
            f"{result.get('label', '')} — "
            f"{result.get('description', '')}"
        )

    return results[0]["id"]

In [5]:
qid = find_qid("Albert Einstein")

Q937: Albert Einstein — German-born theoretical physicist (1879–1955)
Q2030894: Albert Einstein College of Medicine — private medical school in New York City, NY
Q2253683: Albert Einstein — train service
Q21200226: Albert Einstein — Wikimedia permanent duplicate item
Q47513150: Albert Einstein — painting by Max Westfield


In [16]:
import httpx
import json

QUERY = f"""
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>

SELECT ?birthday
WHERE {{
    wd:{qid} wdt:P569 ?birthday .
}} """

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "ebruhn/1.0 (mailto:s246070@dtu.dk)"
}

with httpx.Client(timeout=30) as client:
    r = client.get(
        SPARQL_ENDPOINT,
        params={"query": QUERY},
        headers=headers
    )
    r.raise_for_status()
    data = r.json()

print(json.dumps(data, indent=2))

v = lambda row, field: row[field].get("value", "-")

for row in data.get("results", {}).get("bindings", []):
    print(
        f"Birthday: {v(row, 'birthday')}"
    )
data["results"]["bindings"][0]["birthday"]["value"]

{
  "head": {
    "vars": [
      "birthday"
    ]
  },
  "results": {
    "bindings": [
      {
        "birthday": {
          "datatype": "http://www.w3.org/2001/XMLSchema#dateTime",
          "type": "literal",
          "value": "1879-03-14T00:00:00Z"
        }
      }
    ]
  }
}
Birthday: 1879-03-14T00:00:00Z


'1879-03-14T00:00:00Z'